In [2]:
# Mounts Drive - Run once before subsequent blocks

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Objectives

#question tokenization
#simulaity

#Word to vec multiple choice
#for simulaty between answer choices and questions
#sentence embedding (pytorch or huggingface)

#Location of the code

#each keyword is a column and count for each question
# ignore within parenthesis
#https://www.sbert.net/
#https://chatgpt.com/share/689ce03c-de50-8005-a97a-bce53843952c

Wrote 4002 rows to /content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv


In [ ]:
# CSV Metrics Extraction
import os, json, csv, re, keyword

# ---- Paths ----
HPC_Complete = '/content/drive/Shareddrives/Lopez_Morrison_Summer25/HPC_Complete'
JSON_IndividualQs = '/content/drive/Shareddrives/Lopez_Morrison_Summer25/JSON_individualQs'
out_csv = '/content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv'

# ---- Lexical helpers ----
python_keywords = set(keyword.kwlist)
cf_kw = {'if','elif','else','for','while','try','except','finally','with','match','case','and','or'}
_op_re = re.compile(r'(\*\*|//|==|!=|<=|>=|:=|<<|>>|[-+*/%<>=&|^~])')
_ident_tok_re = re.compile(r'\b[A-Za-z_][A-Za-z0-9_]*\b')
_single_ident_line_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')

def is_code_line(line: str) -> bool:
    s = line.strip()
    if not s: return False
    if s.startswith('#'): return False
    if _single_ident_line_re.fullmatch(s): return False  # bare identifier only
    return True

def expand_indent_width(line: str, tabsize: int = 4) -> int:
    expanded = line.expandtabs(tabsize)
    return len(expanded) - len(expanded.lstrip(' '))

def parse_filename(fname: str):
    """
    Expected underscore pattern like:
      set_1_drag-and-drop_If-Statements_gpt-4o_cleaned_q1_hpc.json
    Split indices:
      0:set  1:1  2:method  3:subject  4:model  5:cleaned  6:q1  7:hpc
    """
    name, _ext = os.path.splitext(fname)
    parts = name.split('_')
    try:
        set_number = int(parts[1])
        method = parts[2]
        subject = parts[3]
        model = parts[4]
        q_match = re.fullmatch(r'q(\d+)', parts[6])
        question_number = int(q_match.group(1)) if q_match else None
        return {
            "set_number": set_number,
            "method": method,
            "subject": subject,
            "model": model,
            "question_number": question_number,
            "note": ""
        }
    except Exception as e:
        return {
            "set_number": None,
            "method": "",
            "subject": "",
            "model": "",
            "question_number": None,
            "note": f"PARSE_WARN: {type(e).__name__}"
        }

# ---- Parent JSON processing helpers ----
def _extract_strings(obj):
    """Recursively collect all string values inside a JSON-like object."""
    out = []
    if isinstance(obj, str):
        out.append(obj)
    elif isinstance(obj, dict):
        for v in obj.values():
            out.extend(_extract_strings(v))
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            out.extend(_extract_strings(v))
    return out

def _agg_metrics_over_snippets(snippets):
    """Aggregate code-like metrics over a list of text snippets (parent side)."""
    parent_snippets = 0
    parent_newlines = 0
    parent_chars = 0              # chars across snippets (info only)
    parent_chars_nonws = 0
    parent_pykeywords = 0

    parent_pykeywords_cf = 0
    parent_max_nesting_depth = 0
    parent_tokens = 0
    parent_operators = 0
    code_line_lengths = []
    tokens_per_code_line = []

    for s in snippets:
        if not isinstance(s, str):
            continue
        parent_snippets += 1

        s_edges = s.lstrip('\n').rstrip('\n')
        if '\n' in s_edges:
            parent_newlines += s_edges.count('\n')

        parent_chars += len(s)
        parent_chars_nonws += len(re.sub(r'\s+', '', s))

        toks_all = _ident_tok_re.findall(s)
        parent_pykeywords += sum(1 for t in toks_all if t in python_keywords)

        # Trim outer blank lines
        lines = s.splitlines()
        i0 = 0
        while i0 < len(lines) and not lines[i0].strip():
            i0 += 1
        i1 = len(lines)
        while i1 > i0 and not lines[i1-1].strip():
            i1 -= 1
        lines = lines[i0:i1]

        code_mask = [is_code_line(ln) for ln in lines]

        # lengths & tokens over code lines
        for ln, is_code in zip(lines, code_mask):
            if is_code:
                code_line_lengths.append(len(ln))
                ln_tokens = _ident_tok_re.findall(ln)
                tokens_per_code_line.append(len(ln_tokens))
                parent_tokens += len(ln_tokens)

        code_only = "\n".join(ln for ln, is_code in zip(lines, code_mask) if is_code)
        parent_operators += len(_op_re.findall(code_only))
        cf_tokens = _ident_tok_re.findall(code_only)
        parent_pykeywords_cf += sum(1 for t in cf_tokens if t in cf_kw)

        if lines:
            snippet_max_indent = 0
            for ln, is_code in zip(lines, code_mask):
                if is_code:
                    indent_spaces = expand_indent_width(ln, 4)
                    snippet_max_indent = max(snippet_max_indent, indent_spaces)
            snippet_depth = snippet_max_indent // 4
            parent_max_nesting_depth = max(parent_max_nesting_depth, snippet_depth)

    parent_avg_line_length = (sum(code_line_lengths) / len(code_line_lengths)) if code_line_lengths else 0.0
    parent_max_line_length = max(code_line_lengths) if code_line_lengths else 0
    parent_avg_tokens_per_code_line = (sum(tokens_per_code_line) / len(tokens_per_code_line)) if tokens_per_code_line else 0.0
    parent_operators_per_token_ratio = (parent_operators / max(1, parent_tokens))

    return {
        "parent_snippets": parent_snippets,
        "parent_newlines": parent_newlines,
        "parent_chars_from_snippets": parent_chars,
        "parent_chars_nonws": parent_chars_nonws,
        "parent_pykeywords": parent_pykeywords,
        "parent_pykeywords_cf": parent_pykeywords_cf,
        "parent_max_nesting_depth": parent_max_nesting_depth,
        "parent_tokens": parent_tokens,
        "parent_avg_tokens_per_code_line": round(parent_avg_tokens_per_code_line, 3),
        "parent_operators": parent_operators,
        "parent_operators_per_token_ratio": round(parent_operators_per_token_ratio, 4),
        "parent_avg_line_length": round(parent_avg_line_length, 2),
        "parent_max_line_length": parent_max_line_length,
    }

# ---- Index all parent files (paths, raw length, parsed JSON) ----
parent_file_text_len = {}   # base -> char length of full file text
parent_file_json = {}       # base -> parsed JSON object (or None)
parent_file_path = {}       # base -> (relative_folder_from_JSON_IndividualQs, filename)

for dirpath, _dirnames, filenames in os.walk(JSON_IndividualQs):
    for fname in filenames:
        if not fname.lower().endswith('.json'): continue
        fpath = os.path.join(dirpath, fname)
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                raw = f.read()
            base = os.path.splitext(fname)[0].lower()
            parent_file_text_len[base] = len(raw)
            try:
                parent_file_json[base] = json.loads(raw)
            except Exception:
                parent_file_json[base] = None
            rel_folder = os.path.relpath(dirpath, JSON_IndividualQs)
            if rel_folder == '.':
                rel_folder = ''  # top-level
            parent_file_path[base] = (rel_folder, fname)
        except Exception:
            # Skip unreadable parent files
            continue

# ---- Walk HPC folders and compute metrics ----
rows = []
for hpc_folder in os.listdir(HPC_Complete):
    subfolder = os.path.join(HPC_Complete, hpc_folder)
    if not os.path.isdir(subfolder):
        continue

    # Extraction type by folder name
    if hpc_folder == "JSON_individualQs_HPC_done":
        extraction_type = "LLM"
    elif hpc_folder == "JSON_individualQs_mExtract":
        extraction_type = "Manual"
    else:
        extraction_type = ""

    for hpc_filename in os.listdir(subfolder):
        filepath = os.path.join(subfolder, hpc_filename)
        if not os.path.isfile(filepath) or not hpc_filename.lower().endswith('.json'):
            continue

        meta = parse_filename(hpc_filename)

        # Parent mapping: strip trailing "_hpc" from HPC base
        hpc_base = os.path.splitext(hpc_filename)[0]
        parent_key = (hpc_base[:-4] if hpc_base.endswith('_hpc') else hpc_base).lower()

        # Parent path + raw length + metrics over strings
        parent_chars = parent_file_text_len.get(parent_key, 0)
        parent_folder, parent_filename = parent_file_path.get(parent_key, ("", ""))
        parent_metrics = {
            "parent_snippets": 0,
            "parent_newlines": 0,
            "parent_chars_from_snippets": 0,
            "parent_chars_nonws": 0,
            "parent_pykeywords": 0,
            "parent_pykeywords_cf": 0,
            "parent_max_nesting_depth": 0,
            "parent_tokens": 0,
            "parent_avg_tokens_per_code_line": 0.0,
            "parent_operators": 0,
            "parent_operators_per_token_ratio": 0.0,
            "parent_avg_line_length": 0.0,
            "parent_max_line_length": 0,
        }
        pj = parent_file_json.get(parent_key, None)
        if pj is not None:
            parent_snippet_texts = _extract_strings(pj)
            parent_metrics = _agg_metrics_over_snippets(parent_snippet_texts)

        # ---- Load HPC and compute HPC metrics ----
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            # If HPC JSON can't load, write a row with zeros for HPC metrics
            rows.append({
                "hpc_folder": hpc_folder,
                "hpc_filename": hpc_filename,
                "parent_folder": parent_folder,
                "parent_filename": parent_filename,
                "HPC_extraction_type": extraction_type,
                **meta,

                # HPC metrics (zeros)
                "hpc_snippets": 0,
                "hpc_newlines": 0,
                "hpc_chars": 0,
                "hpc_chars_nonws": 0,
                "hpc_pykeywords": 0,
                "hpc_pykeywords_cf": 0,
                "hpc_max_nesting_depth": 0,
                "hpc_tokens": 0,
                "hpc_avg_tokens_per_code_line": 0.0,
                "hpc_operators": 0,
                "hpc_operators_per_token_ratio": 0.0,
                "hpc_avg_line_length": 0.0,
                "hpc_max_line_length": 0,

                # Parent metrics
                "parent_chars": parent_chars,
                **parent_metrics,

                "note": (meta.get("note", "") + (" | SKIPPED: JSONDecodeError" if isinstance(e, json.JSONDecodeError) else f" | SKIPPED: {type(e).__name__}")).strip(" |")
            })
            continue

        code_list = data.get("HPC", [])
        if not isinstance(code_list, list):
            code_list = []

        hpc_snippets = 0
        hpc_newlines = 0
        hpc_chars = 0
        hpc_chars_nonws = 0
        hpc_pykeywords = 0
        hpc_pykeywords_cf = 0
        hpc_max_nesting_depth = 0
        hpc_tokens = 0
        hpc_operators = 0
        code_line_lengths = []
        tokens_per_code_line = []

        for code in code_list:
            if not isinstance(code, str):
                continue
            hpc_snippets += 1

            code_edges = code.lstrip('\n').rstrip('\n')
            if '\n' in code_edges:
                hpc_newlines += code_edges.count('\n')

            hpc_chars += len(code)
            hpc_chars_nonws += len(re.sub(r'\s+', '', code))

            toks_all = _ident_tok_re.findall(code)
            hpc_pykeywords += sum(1 for t in toks_all if t in python_keywords)

            # Trim outer blank lines only
            lines = code.splitlines()
            i0 = 0
            while i0 < len(lines) and not lines[i0].strip():
                i0 += 1
            i1 = len(lines)
            while i1 > i0 and not lines[i1-1].strip():
                i1 -= 1
            lines = lines[i0:i1]

            code_mask = [is_code_line(ln) for ln in lines]

            for ln, is_code in zip(lines, code_mask):
                if is_code:
                    code_line_lengths.append(len(ln))
                    ln_tokens = _ident_tok_re.findall(ln)
                    tokens_per_code_line.append(len(ln_tokens))
                    hpc_tokens += len(ln_tokens)

            code_only = "\n".join(ln for ln, is_code in zip(lines, code_mask) if is_code)
            hpc_operators += len(_op_re.findall(code_only))

            cf_tokens = _ident_tok_re.findall(code_only)
            hpc_pykeywords_cf += sum(1 for t in cf_tokens if t in cf_kw)

            if lines:
                snippet_max_indent = 0
                for ln, is_code in zip(lines, code_mask):
                    if is_code:
                        indent_spaces = expand_indent_width(ln, 4)
                        snippet_max_indent = max(snippet_max_indent, indent_spaces)
                snippet_depth = snippet_max_indent // 4
                hpc_max_nesting_depth = max(hpc_max_nesting_depth, snippet_depth)

        hpc_avg_line_length = (sum(code_line_lengths) / len(code_line_lengths)) if code_line_lengths else 0.0
        hpc_max_line_length = max(code_line_lengths) if code_line_lengths else 0
        hpc_avg_tokens_per_code_line = (sum(tokens_per_code_line) / len(tokens_per_code_line)) if tokens_per_code_line else 0.0
        hpc_operators_per_token_ratio = (hpc_operators / max(1, hpc_tokens))

        rows.append({
            "hpc_folder": hpc_folder,
            "hpc_filename": hpc_filename,
            "parent_folder": parent_folder,
            "parent_filename": parent_filename,
            "HPC_extraction_type": extraction_type,
            **meta,

            # HPC metrics
            "hpc_snippets": hpc_snippets,
            "hpc_newlines": hpc_newlines,
            "hpc_chars": hpc_chars,
            "hpc_chars_nonws": hpc_chars_nonws,
            "hpc_pykeywords": hpc_pykeywords,
            "hpc_pykeywords_cf": hpc_pykeywords_cf,
            "hpc_max_nesting_depth": hpc_max_nesting_depth,
            "hpc_tokens": hpc_tokens,
            "hpc_avg_tokens_per_code_line": round(hpc_avg_tokens_per_code_line, 3),
            "hpc_operators": hpc_operators,
            "hpc_operators_per_token_ratio": round(hpc_operators_per_token_ratio, 4),
            "hpc_avg_line_length": round(hpc_avg_line_length, 2),
            "hpc_max_line_length": hpc_max_line_length,

            # Parent metrics
            "parent_chars": parent_chars,  # raw file char length
            **parent_metrics,

            "note": meta.get("note", "")
        })

# ---- CSV schema ----
fieldnames = [
    # Paths
    "hpc_folder", "hpc_filename", "parent_folder", "parent_filename",
    # Meta
    "HPC_extraction_type", "set_number", "method", "subject", "model", "question_number",

    # HPC metrics
    "hpc_snippets", "hpc_newlines",
    "hpc_chars", "hpc_chars_nonws", "hpc_pykeywords",
    "hpc_pykeywords_cf", "hpc_max_nesting_depth",
    "hpc_tokens", "hpc_avg_tokens_per_code_line",
    "hpc_operators", "hpc_operators_per_token_ratio",
    "hpc_avg_line_length", "hpc_max_line_length",

    # Parent metrics
    "parent_chars",                      # raw file length (full parent JSON text)
    "parent_snippets", "parent_newlines",
    "parent_chars_from_snippets", "parent_chars_nonws",
    "parent_pykeywords", "parent_pykeywords_cf",
    "parent_max_nesting_depth",
    "parent_tokens", "parent_avg_tokens_per_code_line",
    "parent_operators", "parent_operators_per_token_ratio",
    "parent_avg_line_length", "parent_max_line_length",

    # Notes
    "note"
]

# ---- Write CSV ----
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
with open(out_csv, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.DictWriter(cf, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {out_csv}")


Wrote 4002 rows to /content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv


In [3]:
# CSV Metrics Extraction (with answerchoices handling + MCQ flag)
import os, json, csv, re, keyword

# ---- Paths ----
HPC_Complete = '/content/drive/Shareddrives/Lopez_Morrison_Summer25/HPC_Complete'
JSON_IndividualQs = '/content/drive/Shareddrives/Lopez_Morrison_Summer25/JSON_individualQs'
out_csv = '/content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv'

# ---- Lexical helpers ----
python_keywords = set(keyword.kwlist)
cf_kw = {'if','elif','else','for','while','try','except','finally','with','match','case','and','or'}
_op_re = re.compile(r'(\*\*|//|==|!=|<=|>=|:=|<<|>>|[-+*/%<>=&|^~])')
_ident_tok_re = re.compile(r'\b[A-Za-z_][A-Za-z0-9_]*\b')
_single_ident_line_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')

def is_code_line(line: str) -> bool:
    s = line.strip()
    if not s: return False
    if s.startswith('#'): return False
    if _single_ident_line_re.fullmatch(s): return False  # bare identifier only
    return True

def expand_indent_width(line: str, tabsize: int = 4) -> int:
    expanded = line.expandtabs(tabsize)
    return len(expanded) - len(expanded.lstrip(' '))

def parse_filename(fname: str):
    """
    Expected underscore pattern like:
      set_1_drag-and-drop_If-Statements_gpt-4o_cleaned_q1_hpc.json
    Split indices:
      0:set  1:1  2:method  3:subject  4:model  5:cleaned  6:q1  7:hpc
    """
    name, _ext = os.path.splitext(fname)
    parts = name.split('_')
    try:
        set_number = int(parts[1])
        method = parts[2]
        subject = parts[3]
        model = parts[4]
        q_match = re.fullmatch(r'q(\d+)', parts[6])
        question_number = int(q_match.group(1)) if q_match else None
        return {
            "set_number": set_number,
            "method": method,
            "subject": subject,
            "model": model,
            "question_number": question_number,
            "note": ""
        }
    except Exception as e:
        return {
            "set_number": None, "method": "", "subject": "", "model": "",
            "question_number": None, "note": f"PARSE_WARN: {type(e).__name__}"
        }

# ---- Parent JSON processing helpers ----
def _extract_strings(obj):
    """Recursively collect all string values inside a JSON-like object."""
    out = []
    if isinstance(obj, str):
        out.append(obj)
    elif isinstance(obj, dict):
        for v in obj.values():
            out.extend(_extract_strings(v))
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            out.extend(_extract_strings(v))
    return out

def _agg_metrics_over_snippets(snippets):
    """Aggregate code-like metrics over a list of text snippets (parent side)."""
    parent_snippets = 0
    parent_newlines = 0
    parent_chars = 0              # chars across snippets (info only)
    parent_chars_nonws = 0
    parent_pykeywords = 0

    parent_pykeywords_cf = 0
    parent_max_nesting_depth = 0
    parent_tokens = 0
    parent_operators = 0
    code_line_lengths = []
    tokens_per_code_line = []

    for s in snippets:
        if not isinstance(s, str):
            continue
        parent_snippets += 1

        s_edges = s.lstrip('\n').rstrip('\n')
        if '\n' in s_edges:
            parent_newlines += s_edges.count('\n')

        parent_chars += len(s)
        parent_chars_nonws += len(re.sub(r'\s+', '', s))

        toks_all = _ident_tok_re.findall(s)
        parent_pykeywords += sum(1 for t in toks_all if t in python_keywords)

        # Trim outer blank lines
        lines = s.splitlines()
        i0 = 0
        while i0 < len(lines) and not lines[i0].strip():
            i0 += 1
        i1 = len(lines)
        while i1 > i0 and not lines[i1-1].strip():
            i1 -= 1
        lines = lines[i0:i1]

        code_mask = [is_code_line(ln) for ln in lines]

        # lengths & tokens over code lines
        for ln, is_code in zip(lines, code_mask):
            if is_code:
                code_line_lengths.append(len(ln))
                ln_tokens = _ident_tok_re.findall(ln)
                tokens_per_code_line.append(len(ln_tokens))
                parent_tokens += len(ln_tokens)

        code_only = "\n".join(ln for ln, is_code in zip(lines, code_mask) if is_code)
        parent_operators += len(_op_re.findall(code_only))
        cf_tokens = _ident_tok_re.findall(code_only)
        parent_pykeywords_cf += sum(1 for t in cf_tokens if t in cf_kw)

        if lines:
            snippet_max_indent = 0
            for ln, is_code in zip(lines, code_mask):
                if is_code:
                    indent_spaces = expand_indent_width(ln, 4)
                    snippet_max_indent = max(snippet_max_indent, indent_spaces)
            snippet_depth = snippet_max_indent // 4
            parent_max_nesting_depth = max(parent_max_nesting_depth, snippet_depth)

    parent_avg_line_length = (sum(code_line_lengths) / len(code_line_lengths)) if code_line_lengths else 0.0
    parent_max_line_length = max(code_line_lengths) if code_line_lengths else 0
    parent_avg_tokens_per_code_line = (sum(tokens_per_code_line) / len(tokens_per_code_line)) if tokens_per_code_line else 0.0
    parent_operators_per_token_ratio = (parent_operators / max(1, parent_tokens))

    return {
        "parent_snippets": parent_snippets,
        "parent_newlines": parent_newlines,
        "parent_chars_from_snippets": parent_chars,
        "parent_chars_nonws": parent_chars_nonws,
        "parent_pykeywords": parent_pykeywords,
        "parent_pykeywords_cf": parent_pykeywords_cf,
        "parent_max_nesting_depth": parent_max_nesting_depth,
        "parent_tokens": parent_tokens,
        "parent_avg_tokens_per_code_line": round(parent_avg_tokens_per_code_line, 3),
        "parent_operators": parent_operators,
        "parent_operators_per_token_ratio": round(parent_operators_per_token_ratio, 4),
        "parent_avg_line_length": round(parent_avg_line_length, 2),
        "parent_max_line_length": parent_max_line_length,
    }

def _extract_answerchoices_stats(parent_json_obj):
    """
    Extract number of answer choices and average choice length (chars) from a parent JSON.
    Supports keys:
      - "answers"  (list|dict)
      - "choices"  (list|dict)
      - "options"  (list|dict)
    Handles list/dict items that are strings or dicts (e.g., {"text": "..."}).
    Returns (count, avg_len)
    """
    def collect_texts(container):
        texts_local = []

        def take_text(v):
            if isinstance(v, str):
                t = v.strip()
                if t:
                    texts_local.append(t)
            elif isinstance(v, dict):
                # Prefer common fields; fall back to any string value
                picked = False
                for key in ("text", "label", "value", "option", "name", "content"):
                    if key in v and isinstance(v[key], str):
                        t = v[key].strip()
                        if t:
                            texts_local.append(t)
                            picked = True
                            break
                if not picked:
                    for vv in v.values():
                        if isinstance(vv, str):
                            t = vv.strip()
                            if t:
                                texts_local.append(t)

        if isinstance(container, list):
            for item in container:
                take_text(item)
        elif isinstance(container, dict):
            for v in container.values():
                take_text(v)

        return texts_local

    if not isinstance(parent_json_obj, dict):
        return 0, 0.0

    texts = []
    for key in ("answers", "choices", "options"):
        if key in parent_json_obj:
            texts += collect_texts(parent_json_obj[key])

    # de-dup identical strings while preserving order
    seen = set()
    deduped = []
    for t in texts:
        if t not in seen:
            seen.add(t)
            deduped.append(t)

    count = len(deduped)
    avg_len = (sum(len(t) for t in deduped) / count) if count > 0 else 0.0
    return count, round(avg_len, 2)

# ---- Index all parent files (paths, raw length, parsed JSON) ----
parent_file_text_len = {}   # base -> char length of full file text
parent_file_json = {}       # base -> parsed JSON object (or None)
parent_file_path = {}       # base -> (relative_folder_from_JSON_IndividualQs, filename)

for dirpath, _dirnames, filenames in os.walk(JSON_IndividualQs):
    for fname in filenames:
        if not fname.lower().endswith('.json'): continue
        fpath = os.path.join(dirpath, fname)
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                raw = f.read()
            base = os.path.splitext(fname)[0].lower()
            parent_file_text_len[base] = len(raw)
            try:
                parent_file_json[base] = json.loads(raw)
            except Exception:
                parent_file_json[base] = None
            rel_folder = os.path.relpath(dirpath, JSON_IndividualQs)
            if rel_folder == '.':
                rel_folder = ''  # top-level
            parent_file_path[base] = (rel_folder, fname)
        except Exception:
            continue

# ---- Walk HPC folders and compute metrics ----
rows = []
for hpc_folder in os.listdir(HPC_Complete):
    subfolder = os.path.join(HPC_Complete, hpc_folder)
    if not os.path.isdir(subfolder):
        continue

    # Extraction type by folder name
    if hpc_folder == "JSON_individualQs_HPC_done":
        extraction_type = "LLM"
    elif hpc_folder == "JSON_individualQs_mExtract":
        extraction_type = "Manual"
    else:
        extraction_type = ""

    for hpc_filename in os.listdir(subfolder):
        filepath = os.path.join(subfolder, hpc_filename)
        if not os.path.isfile(filepath) or not hpc_filename.lower().endswith('.json'):
            continue

        meta = parse_filename(hpc_filename)

        # Parent mapping: strip trailing "_hpc" from HPC base
        hpc_base = os.path.splitext(hpc_filename)[0]
        parent_key = (hpc_base[:-4] if hpc_base.endswith('_hpc') else hpc_base).lower()

        # Parent path + raw length + metrics over strings + answerchoices stats
        parent_chars = parent_file_text_len.get(parent_key, 0)
        parent_folder, parent_filename = parent_file_path.get(parent_key, ("", ""))
        parent_metrics = {
            "parent_snippets": 0,
            "parent_newlines": 0,
            "parent_chars_from_snippets": 0,
            "parent_chars_nonws": 0,
            "parent_pykeywords": 0,
            "parent_pykeywords_cf": 0,
            "parent_max_nesting_depth": 0,
            "parent_tokens": 0,
            "parent_avg_tokens_per_code_line": 0.0,
            "parent_operators": 0,
            "parent_operators_per_token_ratio": 0.0,
            "parent_avg_line_length": 0.0,
            "parent_max_line_length": 0,
        }
        parent_answerchoices_count = 0
        parent_answerchoices_avg_len = 0.0

        pj = parent_file_json.get(parent_key, None)
        if pj is not None:
            parent_snippet_texts = _extract_strings(pj)
            parent_metrics = _agg_metrics_over_snippets(parent_snippet_texts)
            parent_answerchoices_count, parent_answerchoices_avg_len = _extract_answerchoices_stats(pj)

        # ---- Load HPC and compute HPC metrics ----
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            # If HPC JSON can't load, write a row with zeros for HPC metrics
            note_text = meta.get("note", "")
            if meta.get("method") == "multiple-choice" and (parent_answerchoices_count == 0 or parent_answerchoices_avg_len == 0.0):
                note_text = (note_text + " | " if note_text else "") + "ANSWERCHOICES_NOT_CAPTURED"
            rows.append({
                "hpc_folder": hpc_folder,
                "hpc_filename": hpc_filename,
                "parent_folder": parent_folder,
                "parent_filename": parent_filename,
                "HPC_extraction_type": extraction_type,
                **meta,

                # HPC metrics (zeros)
                "hpc_snippets": 0,
                "hpc_newlines": 0,
                "hpc_chars": 0,
                "hpc_chars_nonws": 0,
                "hpc_pykeywords": 0,
                "hpc_pykeywords_cf": 0,
                "hpc_max_nesting_depth": 0,
                "hpc_tokens": 0,
                "hpc_avg_tokens_per_code_line": 0.0,
                "hpc_operators": 0,
                "hpc_operators_per_token_ratio": 0.0,
                "hpc_avg_line_length": 0.0,
                "hpc_max_line_length": 0,

                # Parent metrics + answerchoices
                "parent_chars": parent_chars,  # raw file char length
                **parent_metrics,
                "parent_answerchoices_count": parent_answerchoices_count,
                "parent_answerchoices_avg_len": parent_answerchoices_avg_len,

                "note": note_text
            })
            continue

        code_list = data.get("HPC", [])
        if not isinstance(code_list, list):
            code_list = []

        hpc_snippets = 0
        hpc_newlines = 0
        hpc_chars = 0
        hpc_chars_nonws = 0
        hpc_pykeywords = 0
        hpc_pykeywords_cf = 0
        hpc_max_nesting_depth = 0
        hpc_tokens = 0
        hpc_operators = 0
        code_line_lengths = []
        tokens_per_code_line = []

        for code in code_list:
            if not isinstance(code, str):
                continue
            hpc_snippets += 1

            code_edges = code.lstrip('\n').rstrip('\n')
            if '\n' in code_edges:
                hpc_newlines += code_edges.count('\n')

            hpc_chars += len(code)
            hpc_chars_nonws += len(re.sub(r'\s+', '', code))

            toks_all = _ident_tok_re.findall(code)
            hpc_pykeywords += sum(1 for t in toks_all if t in python_keywords)

            # Trim outer blank lines only
            lines = code.splitlines()
            i0 = 0
            while i0 < len(lines) and not lines[i0].strip():
                i0 += 1
            i1 = len(lines)
            while i1 > i0 and not lines[i1-1].strip():
                i1 -= 1
            lines = lines[i0:i1]

            code_mask = [is_code_line(ln) for ln in lines]

            for ln, is_code in zip(lines, code_mask):
                if is_code:
                    code_line_lengths.append(len(ln))
                    ln_tokens = _ident_tok_re.findall(ln)
                    tokens_per_code_line.append(len(ln_tokens))
                    hpc_tokens += len(ln_tokens)

            code_only = "\n".join(ln for ln, is_code in zip(lines, code_mask) if is_code)
            hpc_operators += len(_op_re.findall(code_only))

            cf_tokens = _ident_tok_re.findall(code_only)
            hpc_pykeywords_cf += sum(1 for t in cf_tokens if t in cf_kw)

            if lines:
                snippet_max_indent = 0
                for ln, is_code in zip(lines, code_mask):
                    if is_code:
                        indent_spaces = expand_indent_width(ln, 4)
                        snippet_max_indent = max(snippet_max_indent, indent_spaces)
                snippet_depth = snippet_max_indent // 4
                hpc_max_nesting_depth = max(hpc_max_nesting_depth, snippet_depth)

        hpc_avg_line_length = (sum(code_line_lengths) / len(code_line_lengths)) if code_line_lengths else 0.0
        hpc_max_line_length = max(code_line_lengths) if code_line_lengths else 0
        hpc_avg_tokens_per_code_line = (sum(tokens_per_code_line) / len(tokens_per_code_line)) if tokens_per_code_line else 0.0
        hpc_operators_per_token_ratio = (hpc_operators / max(1, hpc_tokens))

        # Note flag for missing answerchoices capture (multiple-choice only)
        row_note = meta.get("note", "")
        if meta.get("method") == "multiple-choice" and (parent_answerchoices_count == 0 or parent_answerchoices_avg_len == 0.0):
            row_note = (row_note + " | " if row_note else "") + "ANSWERCHOICES_NOT_CAPTURED"

        rows.append({
            "hpc_folder": hpc_folder,
            "hpc_filename": hpc_filename,
            "parent_folder": parent_folder,
            "parent_filename": parent_filename,
            "HPC_extraction_type": extraction_type,
            **meta,

            # HPC metrics
            "hpc_snippets": hpc_snippets,
            "hpc_newlines": hpc_newlines,
            "hpc_chars": hpc_chars,
            "hpc_chars_nonws": hpc_chars_nonws,
            "hpc_pykeywords": hpc_pykeywords,
            "hpc_pykeywords_cf": hpc_pykeywords_cf,
            "hpc_max_nesting_depth": hpc_max_nesting_depth,
            "hpc_tokens": hpc_tokens,
            "hpc_avg_tokens_per_code_line": round(hpc_avg_tokens_per_code_line, 3),
            "hpc_operators": hpc_operators,
            "hpc_operators_per_token_ratio": round(hpc_operators_per_token_ratio, 4),
            "hpc_avg_line_length": round(hpc_avg_line_length, 2),
            "hpc_max_line_length": hpc_max_line_length,

            # Parent metrics + answerchoices
            "parent_chars": parent_chars,  # raw file length
            **parent_metrics,
            "parent_answerchoices_count": parent_answerchoices_count,
            "parent_answerchoices_avg_len": parent_answerchoices_avg_len,

            "note": row_note
        })

# ---- CSV schema ----
fieldnames = [
    # Paths
    "hpc_folder", "hpc_filename", "parent_folder", "parent_filename",
    # Meta
    "HPC_extraction_type", "set_number", "method", "subject", "model", "question_number",

    # HPC metrics
    "hpc_snippets", "hpc_newlines",
    "hpc_chars", "hpc_chars_nonws", "hpc_pykeywords",
    "hpc_pykeywords_cf", "hpc_max_nesting_depth",
    "hpc_tokens", "hpc_avg_tokens_per_code_line",
    "hpc_operators", "hpc_operators_per_token_ratio",
    "hpc_avg_line_length", "hpc_max_line_length",

    # Parent metrics
    "parent_chars",
    "parent_snippets", "parent_newlines",
    "parent_chars_from_snippets", "parent_chars_nonws",
    "parent_pykeywords", "parent_pykeywords_cf",
    "parent_max_nesting_depth",
    "parent_tokens", "parent_avg_tokens_per_code_line",
    "parent_operators", "parent_operators_per_token_ratio",
    "parent_avg_line_length", "parent_max_line_length",

    # Parent answerchoices (captures answers / choices / options)
    "parent_answerchoices_count", "parent_answerchoices_avg_len",

    # Notes
    "note"
]

# ---- Write CSV ----
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
with open(out_csv, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.DictWriter(cf, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {out_csv}")


Wrote 4002 rows to /content/drive/Shareddrives/Lopez_Morrison_Summer25/code_counts/code_counts.csv
